# 03 - HTTP Status Codes

## 1. Status Code Families

**Definition:** A status code is a 3-digit number the server sends back to say **how the request went**.

| Family | Meaning | Think of it as |
|---|---|---|
| 1xx | Informational | "Hold on" (rarely used) |
| 2xx | Success | "Done" |
| 3xx | Redirection | "Look somewhere else" |
| 4xx | Client error | "You made a mistake" |
| 5xx | Server error | "We made a mistake" |

**Real-world example:** On a shopping site, opening your orders returns 200. Opening a product that was removed returns 404. If the server crashes during checkout, you get 500.

**Interview line:** "2xx means success, 3xx means redirect, 4xx means the client sent something wrong, and 5xx means the server failed."

## 2. Common Status Codes

| Code | Name | When to use | Example |
|---|---|---|---|
| 200 | OK | Successful GET, PUT, PATCH | GET /orders/5 returns the order |
| 201 | Created | Successful POST that creates something | POST /orders creates order 6 |
| 204 | No Content | Success, nothing to return | DELETE /orders/5 |
| 301 | Moved Permanently | Resource has a new URL forever | Old /v1/products moved to /v2/products |
| 302 | Found (temporary redirect) | Temporary move | Redirect to login page |
| 304 | Not Modified | Cached copy is still valid | Browser reuses cached product image |
| 400 | Bad Request | Invalid input | Quantity is -2 |
| 401 | Unauthorized | Not logged in / bad or missing token | No JWT sent |
| 403 | Forbidden | Logged in but not allowed | Customer tries an admin endpoint |
| 404 | Not Found | Resource does not exist | GET /orders/999 |
| 409 | Conflict | Clashes with current state | Not enough stock, or duplicate email |
| 422 | Unprocessable Entity | Right format, failed validation | FastAPI: missing required field |
| 429 | Too Many Requests | Rate limit hit | 100+ calls in a minute |
| 500 | Internal Server Error | Unhandled bug | Code crashes |
| 502 | Bad Gateway | Upstream service gave a bad reply | Payment service is down |
| 503 | Service Unavailable | Server overloaded or in maintenance | Site down for deployment |

**401 vs 403:** 401 = "Who are you?" (authentication). 403 = "I know you, but you can't do this" (authorization).

**400 vs 422:** 400 = request is broken. 422 = request is readable but the data fails validation. FastAPI returns 422 by default for Pydantic errors.

**Interview line:** "I return 201 for creates, 204 for deletes, 401 when the token is missing, 403 when permission is missing, 409 for conflicts like low stock, and never leak details on a 500."

In [ ]:
# ==================================================
# 3. PICKING THE RIGHT STATUS CODE - order API
# ==================================================
stock = {"laptop": 2}
orders = {5: {"item": "laptop", "qty": 1}}

def place_order(token, role, item, qty):
    if token is None:
        return 401, "Missing token"
    if role != "customer":
        return 403, "Only customers can place orders"
    if qty <= 0:
        return 400, "Quantity must be positive"
    if item not in stock:
        return 404, "Product not found"
    if stock[item] < qty:
        return 409, "Not enough stock"
    stock[item] -= qty
    new_id = max(orders) + 1
    orders[new_id] = {"item": item, "qty": qty}
    return 201, f"Order {new_id} created"

tests = [
    ("No token",       None,  "customer", "laptop", 1),
    ("Wrong role",     "jwt", "guest",    "laptop", 1),
    ("Bad quantity",   "jwt", "customer", "laptop", -2),
    ("Unknown item",   "jwt", "customer", "phone",  1),
    ("Too many",       "jwt", "customer", "laptop", 5),
    ("Valid order",    "jwt", "customer", "laptop", 1),
]

print("=" * 55)
for label, token, role, item, qty in tests:
    code, msg = place_order(token, role, item, qty)
    print(f"[{code}] {label:14} -> {msg}")
print("=" * 55)

## Key Takeaways

- 2xx success, 3xx redirect, 4xx client error, 5xx server error
- 201 for create, 204 for delete with no body
- 401 = not authenticated, 403 = not authorized
- 400 = broken request, 422 = validation failed (FastAPI default)
- 409 = conflict with current state, 429 = rate limited
- Never expose stack traces on a 500